# road-damage-shift — Q3b (exploratoire) : peut-on sauver l'INT8 ?

En Q3, l'INT8 (toutes les convolutions, calibration sur 200 images sources) perdait −0,036 de mAP@50 sur le test source
et −0,145 aux États-Unis. Plan 2 × 2, fixé dans `configs/default.yaml` (`deploy.int8_variants`) **après Q3 et avant
tout résultat Q3b** :
- quantifier toute la tête de détection ou la laisser en FP32 (`/model.23/` : 25 convolutions, dont DFL) ;
- calibrer sur 200 images sources, ou sur 100 sources + 100 images américaines **non annotées** (réserve few-shot,
  jamais le test).

`int8_all_src` reproduit Q3 : c'est le contrôle. Référence : ONNX FP32, même entrée 640 × 640.

**Réglages Kaggle** : *Accelerator* = **None**, Internet On, *Add Input* = le dataset des poids S1.
`SMOKE = True` d'abord (≈ 15 min), puis `SMOKE = False` via *Save Version* (≈ 1 h 30).

In [ ]:
# ---- Paramètres -------------------------------------------------------------
REPO_URL = "https://github.com/<TON_USER>/road-damage-shift.git"  # à remplacer
REPO_REF = "main"
REPO_DIR = "/tmp/road-damage-shift"
RAW = "/tmp/rdd_raw"
DATA_OUT = "/tmp/rdd_yolo"
SMOKE = True                 # True : 50 images par jeu d'évaluation, 20 de calibration, 30 passes de latence
S1_MANIFEST_SHA256 = "aff1da5af0e1c61885e4a59ddc058d8f659638117f99ab5d1f5df525cc57e2da"
WEIGHTS = "/kaggle/input/datasets/<TON_USER>/rds-s1-weights/yolo11s_best.pt"  # poids S1 (fichier ou dossier)

In [ ]:
!rm -rf {REPO_DIR} && git clone -q --depth 1 -b {REPO_REF} {REPO_URL} {REPO_DIR}
!pip install -q -e "{REPO_DIR}[train]" onnxslim faster-coco-eval
!nproc; lscpu | grep -E "Model name|^CPU\(s\)|Flags" | cut -c1-200

In [ ]:
import hashlib, json, os, platform, shutil, subprocess, sys, time
from pathlib import Path

sys.path.insert(0, f"{REPO_DIR}/src")
import mlflow, onnx, onnxruntime, pandas as pd, torch, ultralytics, yaml
from ultralytics import YOLO, settings

from roadshift.deploy.bench import bench, cpu_name
from roadshift.deploy.quantize import calibration_mix, quantize_int8, quantized_convs
from roadshift.eval.compare import compare
from roadshift.eval.report import report

CFG = yaml.safe_load(open(f"{REPO_DIR}/configs/default.yaml"))
DEP, CLASSES = CFG["deploy"], CFG["data"]["classes"]
Q3B = DEP["int8_variants"]
GIT_SHA = subprocess.check_output(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], text=True).strip()
VERSIONS = {"torch": torch.__version__, "ultralytics": ultralytics.__version__, "onnx": onnx.__version__,
            "onnxruntime": onnxruntime.__version__, "cpu": cpu_name(), "cpu_count": os.cpu_count()}
print(GIT_SHA[:10], VERSIONS)
W = Path(WEIGHTS)
if W.is_dir():
    pts = sorted(W.rglob("*.pt")); assert len(pts) == 1, f"{W} contient {len(pts)} .pt"; W = pts[0]
W_SHA = hashlib.sha256(W.read_bytes()).hexdigest()
print("poids S1 :", W, W_SHA[:12])

## 0–1. Données (mêmes découpages qu'en S1, vérifiés par empreinte)

In [ ]:
!python -m roadshift.data.download --out {RAW} --countries Japan India Czech United_States
!python -m roadshift.data.convert --config {REPO_DIR}/configs/default.yaml --raw-root {RAW} --out {DATA_OUT} --overwrite
DATA = Path(DATA_OUT)
DATASET_INFO = json.load(open(DATA / "dataset_info.json"))
assert DATASET_INFO["manifest_sha256"] == S1_MANIFEST_SHA256, "Découpages différents de S1 : comparaison invalide"

EVAL = DATA
if SMOKE:  # mini-copie : 50 premières images de chaque liste d'évaluation, mêmes noms de fichiers
    EVAL = Path("/tmp/rdd_yolo_smoke"); shutil.rmtree(EVAL, ignore_errors=True); EVAL.mkdir()
    for d in ("images", "labels"):
        (EVAL / d).symlink_to(DATA / d)
    shutil.copy(DATA / "manifest.csv", EVAL)
    for f in DATA.glob("list_*.txt"):
        (EVAL / f.name).write_text("\n".join(f.read_text().split()[:50]) + "\n")
    for f in DATA.glob("data_*.yaml"):
        shutil.copy(f, EVAL)
print("évaluation sur :", EVAL)

## 2. Export ONNX FP32 et quatre variantes INT8

In [ ]:
MODELS = Path("/kaggle/working/models"); MODELS.mkdir(exist_ok=True)
pt = MODELS / "yolo11s_s1.pt"; shutil.copy(W, pt)
onnx_fp32 = Path(YOLO(str(pt)).export(format="onnx", imgsz=DEP["imgsz"], opset=DEP["onnx_opset"],
                                         dynamic=False, simplify=True))
onnx_fp32 = onnx_fp32.rename(MODELS / "yolo11s_s1_fp32.onnx")
VARIANTS, AUDIT = {"onnx_fp32": onnx_fp32}, []
for name, v in Q3B.items():
    spec = {k: (max(1, n // 10) if SMOKE else n) for k, n in v["calibration"].items()}
    calib = calibration_mix(DATA, spec, CFG["project"]["seed"])
    t0 = time.time()
    out = quantize_int8(onnx_fp32, MODELS / f"yolo11s_s1_{name}.onnx", calib,
                        {**DEP["int8"], "exclude_name_contains": v["exclude_name_contains"]}, DEP["imgsz"])
    n, q, fp = quantized_convs(out)
    AUDIT.append({"variant": name, "calibration": json.dumps(spec), "n_calibration": len(calib),
                  "conv_total": n, "conv_int8": q, "conv_fp32": len(fp), "quant_s": round(time.time() - t0, 1),
                  "size_MB": round(out.stat().st_size / 1e6, 2)})
    VARIANTS[name] = out
AUDIT = pd.DataFrame(AUDIT)
display(AUDIT)
assert AUDIT.set_index("variant").loc["int8_all_src", "conv_fp32"] == 0
assert (AUDIT[AUDIT.variant.str.contains("nohead")].conv_fp32 > 0).all()

## 3. Latence CPU du modèle seul (même tenseur, médiane et 90e centile)

In [ ]:
lat = DEP["latency"]
threads = min(lat["threads"], os.cpu_count())
LAT = pd.DataFrame(bench(list(VARIANTS.values()), DEP["imgsz"], threads, 5 if SMOKE else lat["warmup"],
                         30 if SMOKE else lat["runs"]))
LAT.insert(0, "variant", list(VARIANTS))
display(LAT[["variant", "size_MB", "ms_median", "ms_p90", "fps_median", "threads", "cpu"]])

## 4. Précision : évaluation des cinq versions (CPU), IC bootstrap et écarts appariés
Le seuil du F1 de chaque version est choisi sur sa propre validation source. Latence de bout en bout
(prétraitement + inférence + post-traitement) relevée par Ultralytics pendant l'évaluation du test source.

In [ ]:
RUNS = Path("/kaggle/working/runs_q3b")
EVALS = {"source_val": ("data_source.yaml", "val"), "source_test": ("data_source.yaml", "test"),
         "test_United_States": ("data_eval_United_States.yaml", "val")}
E2E = []
t0 = time.time()
for v, p in VARIANTS.items():
    for tag, (yml, split) in EVALS.items():
        r = YOLO(str(p), task="detect").val(data=str(EVAL / yml), split=split, imgsz=DEP["imgsz"],
                                            batch=8 if p.suffix == ".pt" else 1, conf=CFG["eval"]["conf_map"],
                                            save_json=True, plots=False, device="cpu", verbose=False,
                                            project=str(RUNS / v), name=f"eval_{tag}", exist_ok=True)
        if tag == "source_test":
            E2E.append({"variant": v, **{f"e2e_{k}_ms": round(x, 1) for k, x in r.speed.items()}})
    print(f"{v} évalué ({(time.time() - t0) / 60:.1f} min)")
E2E = pd.DataFrame(E2E)
SCORES = pd.concat([pd.DataFrame(report(RUNS / v, EVAL, v, n_boot=CFG["eval"]["bootstrap"]["n_resamples"]))
                    for v in VARIANTS])
CMP = pd.DataFrame(compare({v: RUNS / v for v in VARIANTS}, EVAL, ["source_test", "test_United_States"],
                           "test_United_States", n_boot=CFG["eval"]["bootstrap"]["n_resamples"]))
display(SCORES[["model", "eval", "n_images", "mAP50", "mAP50_lo", "mAP50_hi", "F1", "F1_lo", "F1_hi"]])
display(CMP[(CMP.comparison == "model_paired") & (CMP.a == "onnx_fp32")])
display(E2E)

# Contrôle : int8_all_src doit reproduire l'INT8 de Q3 (mêmes réglages ; résultats Q3 versionnés dans le dépôt)
if not SMOKE:
    q3 = pd.read_csv(f"{REPO_DIR}/results/deploy_scores.csv").set_index(["model", "eval"])
    for tag in ("source_test", "test_United_States"):
        now = SCORES.set_index(["model", "eval"]).loc[("int8_all_src", tag), "mAP50"]
        print(f"contrôle {tag} : Q3 {q3.loc[('onnx_int8', tag), 'mAP50']:.4f}  Q3b {now:.4f}")

## 5. MLflow et sorties

In [ ]:
def _secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None

for k in ("MLFLOW_TRACKING_USERNAME", "MLFLOW_TRACKING_PASSWORD"):
    if (v_ := _secret(k)):
        os.environ[k] = v_
os.chdir("/kaggle/working")
mlflow.set_tracking_uri(CFG["mlflow"]["tracking_uri"] or _secret("MLFLOW_TRACKING_URI") or "sqlite:////kaggle/working/mlflow.db")
mlflow.set_experiment(CFG["mlflow"]["experiment"] + "-deploy-q3b" + ("-smoke" if SMOKE else ""))
settings.update({"mlflow": False})

OUT = Path("/kaggle/working/results"); OUT.mkdir(exist_ok=True)
suffix = "_smoke" if SMOKE else ""
for df, name in ((LAT, "latency"), (SCORES, "scores"), (CMP, "comparisons"), (E2E, "e2e_latency"), (AUDIT, "audit")):
    df.to_csv(OUT / f"q3b_{name}{suffix}.csv", index=False)
with mlflow.start_run(run_name="deploy_q3b_int8" + suffix):
    mlflow.set_tags({"stage": "S3-deploy-q3b", "exploratory": True, "smoke": SMOKE, "git_sha": GIT_SHA,
                     "dataset_sha256": DATASET_INFO["manifest_sha256"], "s1_weights_sha256": W_SHA, **VERSIONS})
    mlflow.log_params({"imgsz": DEP["imgsz"], "opset": DEP["onnx_opset"], "threads": threads,
                       **{f"int8_{k}": str(v) for k, v in DEP["int8"].items()},
                       **{f"variant_{k}": json.dumps(v) for k, v in Q3B.items()}})
    for _, r in LAT.iterrows():
        mlflow.log_metrics({f"{r.variant}/size_MB": r.size_MB, f"{r.variant}/ms_median": r.ms_median,
                            f"{r.variant}/ms_p90": r.ms_p90})
    for _, r in SCORES.iterrows():
        mlflow.log_metrics({f"{r.model}/{r['eval']}/{k}": float(r[k]) for k in ("mAP50", "mAP50_lo", "mAP50_hi", "F1")})
    for f in OUT.glob(f"q3b_*{suffix}.csv"):
        mlflow.log_artifact(str(f), artifact_path="results")
!cd /kaggle/working && rm -f models/yolo11s_s1.pt models/yolo11s_s1_fp32.onnx && zip -qr s3b_int8_outputs.zip results models mlflow.db mlruns 2>/dev/null; ls -lh /kaggle/working

## À me renvoyer
`s3b_int8_outputs.zip` (résultats, quatre modèles INT8, MLflow). Le modèle FP32 n'y est pas : tu l'as déjà (Q3).